# ZKBrain Cross-Platform BCI Research Algorithm Pipeline
> **Technical Support**: ZKTeco Brain-Computer Interface Division
>
> Designed for universities, neuroscience laboratories, and algorithm developers:
> 1. Raw EEG (512Hz) filtering pipeline (0.5-50Hz bandpass + 50Hz notch filter)
> 2. MNE-Python standard EEG data structure assembly and 10-20 electrode (FP1) configuration
> 3. Welch PSD (Power Spectral Density) and 8-band power analysis
> 4. Attention/Meditation 1Hz temporal dynamic feature modeling
> 5. Export to international research standard EDF+ and CSV datasets

In [ ]:
import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import mne

print(f"MNE Version: {mne.__version__}")

## 1. Simulate / Read ZKBrain 512Hz Raw EEG Signal

In [ ]:
# Sample rate 512Hz, duration 10 seconds
sfreq = 512
duration = 10
n_samples = sfreq * duration
t = np.linspace(0, duration, n_samples, endpoint=False)

# Synthesize EEG: 10Hz Alpha, 4Hz Theta, 50Hz powerline noise, and pink noise
alpha_wave = 25 * np.sin(2 * np.pi * 10 * t)
theta_wave = 15 * np.sin(2 * np.pi * 4 * t)
noise_50hz = 12 * np.sin(2 * np.pi * 50 * t)
pink_noise = np.random.normal(0, 8, n_samples)

raw_eeg = alpha_wave + theta_wave + noise_50hz + pink_noise

plt.figure(figsize=(12, 3))
plt.plot(t[:1024], raw_eeg[:1024], color='#0284c7', lw=1)
plt.title("ZKBrain Raw EEG Waveform (First 2 Seconds)")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude (uV)")
plt.grid(True, alpha=0.3)
plt.show()

## 2. 0.5~50Hz Bandpass Filtering & 50Hz Notch Filter (Butterworth & IIR Notch)

In [ ]:
# 1. 50Hz Notch Filter
b_notch, a_notch = signal.iirnotch(50.0, 30.0, sfreq)
notched_eeg = signal.filtfilt(b_notch, a_notch, raw_eeg)

# 2. 0.5 ~ 50Hz 4th-order zero-phase Butterworth bandpass filter
sos = signal.butter(4, [0.5, 50.0], btype='bandpass', fs=sfreq, output='sos')
filtered_eeg = signal.sosfiltfilt(sos, notched_eeg)

plt.figure(figsize=(12, 4))
plt.plot(t[:1024], raw_eeg[:1024], label='Raw Signal (with 50Hz noise)', color='#ef4444', alpha=0.6)
plt.plot(t[:1024], filtered_eeg[:1024], label='Filtered Signal (0.5-50Hz Clean EEG)', color='#16a34a', lw=1.5)
plt.title("EEG Filtering Comparison")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude (uV)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. MNE-Python Data Structure & Welch PSD Spectrum Estimation

In [ ]:
# Assemble into MNE RawArray (channel: standard FP1 forehead placement)
ch_names = ['FP1']
ch_types = ['eeg']
info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=ch_types)

# Convert uV to V (MNE standard input)
data_mne = filtered_eeg.reshape(1, -1) * 1e-6
raw = mne.io.RawArray(data_mne, info)

# Compute Welch PSD
psd = raw.compute_psd(method='welch', fmin=0.5, fmax=50, n_fft=512)
psds, freqs = psd.get_data(return_freqs=True)

plt.figure(figsize=(10, 4))
plt.plot(freqs, 10 * np.log10(psds[0]), color='#7c3aed', lw=2)
plt.axvspan(0.5, 4, color='#e0ffff', alpha=0.4, label='Delta (0.5-4Hz)')
plt.axvspan(4, 8, color='#c7d2fe', alpha=0.4, label='Theta (4-8Hz)')
plt.axvspan(8, 13, color='#fef08a', alpha=0.4, label='Alpha (8-13Hz)')
plt.axvspan(13, 30, color='#fed7aa', alpha=0.4, label='Beta (13-30Hz)')
plt.axvspan(30, 50, color='#fecaca', alpha=0.4, label='Gamma (30-50Hz)')
plt.title("Welch Power Spectral Density (PSD Spectrum) & 8-Band Distribution")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Power Spectral Density (dB)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Export to International Research Datasets (EDF+ / CSV)

In [ ]:
# Export as CSV time series
df_export = pd.DataFrame({
    'timestamp_ms': (t * 1000).astype(int),
    'raw_eeg_uV': filtered_eeg,
    'attention': 78,
    'meditation': 65
})
df_export.to_csv('zkbrain_research_dataset.csv', index=False)
print("Successfully generated zkbrain_research_dataset.csv dataset")

# Export as European Data Format EDF+
mne.export.export_raw('zkbrain_research_dataset.edf', raw, fmt='edf', overwrite=True)
print("Successfully generated zkbrain_research_dataset.edf file (compatible with EEGLAB / BrainVision / Polyman)")